# Transformer Baseline for Synthetic EHR Generation on MIMIC-III

This notebook demonstrates how to train a Transformer-based generative model on MIMIC-III data and generate synthetic patient records.

## Overview
- **Model**: TransformerEHRGenerator (decoder-only transformer, GPT-style)
- **Dataset**: MIMIC-III diagnosis codes
- **Output**: CSV file with columns: `SUBJECT_ID`, `VISIT_NUM`, `ICD9_CODE`

## Setup
Designed for Google Colab with GPU support.

## 1. Environment Setup

In [1]:
import os

# Where to clone from
clone_repo = "https://github.com/ethanrasmussen/PyHealth.git"
clone_branch = "implement_baseline"

# Where to save repo/package
repo_dir = "/content/PyHealth"

if not os.path.exists(repo_dir):
    !git clone -b {clone_branch} {clone_repo} {repo_dir}
%cd /content/PyHealth

# install your repo without letting pip touch torch/cuda stack
%pip install -e . --no-deps

# now install the runtime deps you actually need for this notebook
# %pip install -U --no-cache-dir --force-reinstall "numpy==2.2.0"
%pip install -U "transformers==4.53.2" "tokenizers" "accelerate" "peft"
%pip install -U "pandas" "tqdm" "litdata" "mne" "rdkit"

Cloning into '/content/PyHealth'...
remote: Enumerating objects: 11727, done.
remote: Counting objects: 100% (719/719), done.
remote: Compressing objects: 100% (453/453), done.
remote: Total 11727 (delta 554), reused 267 (delta 266), pack-reused 11008 (from 3)
Receiving objects: 100% (11727/11727), 141.63 MiB | 34.90 MiB/s, done.
Resolving deltas: 100% (7594/7594), done.
/content/PyHealth
Obtaining file:///content/PyHealth
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pyhealth (pyproject.toml) ... done
  Created wheel for pyhealth: filename=pyhealth-2.0.0-py3-none-any.whl size=10559 sha256=67843616f523d031159856d7123232aef9ded9b796d5e80ca1db2e4cea28684b
  Stored in directory: /tmp/pip-ephem-wheel-cache-p5cyjv_g/wheels/1c/98/da/d6e74a692d0be5faeba6025d7302fd470b

In [2]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("WARNING: Running on CPU. Training will be very slow.")
    device = "cpu"

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [3]:
# Mount Google Drive (optional - for persistent storage)
from google.colab import drive
drive.mount('/content/drive')

# Set paths for persistent storage
DRIVE_ROOT = "/content/drive/MyDrive/PyHealth_Synthetic_EHR"
!mkdir -p "{DRIVE_ROOT}"
!mkdir -p "{DRIVE_ROOT}/data"
!mkdir -p "{DRIVE_ROOT}/models"
!mkdir -p "{DRIVE_ROOT}/output"

Mounted at /content/drive


## 2. Configuration

In [4]:
# Configuration parameters
class Config:
    # Paths
    MIMIC_ROOT = f"{DRIVE_ROOT}/data/mimic3"  # Update this to your MIMIC-III path
    OUTPUT_DIR = f"{DRIVE_ROOT}/output"
    MODEL_SAVE_PATH = f"{DRIVE_ROOT}/models/transformer_ehr_best.pth"

    # Dataset parameters
    MIN_VISITS = 2  # Minimum visits per patient
    MAX_VISITS = None  # Maximum visits to include (None = all)

    # Model architecture
    EMBEDDING_DIM = 256
    NUM_HEADS = 8
    NUM_LAYERS = 6
    DIM_FEEDFORWARD = 1024
    DROPOUT = 0.1
    MAX_SEQ_LENGTH = 512

    # Training parameters
    EPOCHS = 5  # Set to 50-80 for production
    BATCH_SIZE = 64  # Reduce to 32 if OOM errors occur
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5

    # Data split
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.1
    TEST_RATIO = 0.1

    # Generation parameters
    NUM_SYNTHETIC_SAMPLES = 1000  # Set to 10000 for production
    MAX_GEN_VISITS = 10
    MAX_CODES_PER_VISIT = 20
    TEMPERATURE = 1.0
    TOP_K = 50
    TOP_P = 0.95

config = Config()
print("Configuration loaded successfully!")
print(f"Training for {config.EPOCHS} epochs")
print(f"Will generate {config.NUM_SYNTHETIC_SAMPLES} synthetic patients")

Configuration loaded successfully!
Training for 5 epochs
Will generate 1000 synthetic patients


## 3. Data Upload

Upload your MIMIC-III data files to the specified directory. You need:
- `ADMISSIONS.csv`
- `DIAGNOSES_ICD.csv`

These files should be placed in the directory specified by `config.MIMIC_ROOT`.

In [5]:
# Check if MIMIC-III files exist
import os

required_files = ['ADMISSIONS.csv', 'DIAGNOSES_ICD.csv']
files_exist = all(os.path.exists(os.path.join(config.MIMIC_ROOT, f)) for f in required_files)

if files_exist:
    print("✓ All required MIMIC-III files found!")
else:
    print("✗ Missing MIMIC-III files. Please upload:")
    for f in required_files:
        path = os.path.join(config.MIMIC_ROOT, f)
        status = "✓" if os.path.exists(path) else "✗"
        print(f"  {status} {f}")
    print(f"\nUpload files to: {config.MIMIC_ROOT}")

✓ All required MIMIC-III files found!


## 4. Load and Preprocess Data

In [8]:
# Import PyHealth modules
from pyhealth.datasets import MIMIC3Dataset
from pyhealth.tasks import SyntheticEHRGenerationMIMIC3
from pyhealth.datasets import split_by_patient, get_dataloader

print("Loading MIMIC-III dataset...")
# Load base dataset
base_dataset = MIMIC3Dataset(
    root=config.MIMIC_ROOT,
    tables=["DIAGNOSES_ICD"],
    # code_mapping=None,  # Use raw ICD9 codes
)

# print(f"Loaded {len(base_dataset.patients)} patients")

# Apply synthetic EHR generation task
print(f"\nApplying task with min_visits={config.MIN_VISITS}...")
task = SyntheticEHRGenerationMIMIC3(
    min_visits=config.MIN_VISITS,
    max_visits=config.MAX_VISITS
)
sample_dataset = base_dataset.set_task(task)

print(f"Created {len(sample_dataset)} samples")

# Split by patient to prevent data leakage
print(f"\nSplitting data: {config.TRAIN_RATIO}/{config.VAL_RATIO}/{config.TEST_RATIO}")
train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset,
    [config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO]
)

print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

Loading MIMIC-III dataset...
No config path provided, using default config


INFO:pyhealth.datasets.mimic3:No config path provided, using default config


Initializing mimic3 dataset from /content/drive/MyDrive/PyHealth_Synthetic_EHR/data/mimic3 (dev mode: False)


INFO:pyhealth.datasets.base_dataset:Initializing mimic3 dataset from /content/drive/MyDrive/PyHealth_Synthetic_EHR/data/mimic3 (dev mode: False)


No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/84ab835b-5cf1-53be-a864-32fe0fab0768


...

Worker 0 finished processing samples.



INFO:pyhealth.datasets.base_dataset:Worker 0 finished processing samples.


Cached processed samples to /root/.cache/pyhealth/84ab835b-5cf1-53be-a864-32fe0fab0768/tasks/SyntheticEHRGenerationMIMIC3_4493a349-057c-5708-8536-128b789c63a5/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Cached processed samples to /root/.cache/pyhealth/84ab835b-5cf1-53be-a864-32fe0fab0768/tasks/SyntheticEHRGenerationMIMIC3_4493a349-057c-5708-8536-128b789c63a5/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


Created 7499 samples

Splitting data: 0.8/0.1/0.1
Train: 5999 samples
Val: 750 samples
Test: 750 samples


In [9]:
# Create data loaders
print("Creating data loaders...")
train_loader = get_dataloader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True
)
val_loader = get_dataloader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)
test_loader = get_dataloader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Creating data loaders...
Train batches: 94
Val batches: 12
Test batches: 12


In [10]:
# Inspect a sample batch
sample_batch = next(iter(train_loader))
print("Sample batch structure:")
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"  {key}: {type(value)}")

Sample batch structure:
  patient_id: <class 'list'>
  visit_codes: shape torch.Size([64, 7, 39]), dtype torch.int64
  future_codes: shape torch.Size([64, 7, 39]), dtype torch.int64


## 5. Initialize Model

In [11]:
from pyhealth.models import TransformerEHRGenerator

print("Initializing TransformerEHRGenerator...")
model = TransformerEHRGenerator(
    dataset=sample_dataset,
    embedding_dim=config.EMBEDDING_DIM,
    num_heads=config.NUM_HEADS,
    num_layers=config.NUM_LAYERS,
    dim_feedforward=config.DIM_FEEDFORWARD,
    dropout=config.DROPOUT,
    max_seq_length=config.MAX_SEQ_LENGTH
)

# Move model to device
model = model.to(device)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel initialized successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Vocabulary size: {model.vocab_size}")
print(f"Device: {device}")

/content/PyHealth/pyhealth/metrics/calibration.py:122: SyntaxWarning: invalid escape sequence '\c'
  accuracy of 1. Thus, the ECE is :math:`\\frac{1}{3} \cdot 0.49 + \\frac{2}{3}\cdot 0.3=0.3633`.


Initializing TransformerEHRGenerator...

Model initialized successfully!
Total parameters: 8,957,717
Trainable parameters: 8,957,717
Vocabulary size: 4882
Device: cuda


## 6. Training

In [12]:
from pyhealth.trainer import Trainer

print(f"Starting training for {config.EPOCHS} epochs...\n")

# Initialize trainer
trainer = Trainer(
    model=model,
    device=device,
    output_path=config.OUTPUT_DIR,
    exp_name="transformer_ehr_generator"
)

# Train model
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=config.EPOCHS,
    monitor="loss",
    monitor_criterion="min",
    load_best_model_at_last=True
)

print("\n" + "="*50)
print("Training completed!")
print("="*50)

Starting training for 5 epochs...

TransformerEHRGenerator(
  (token_embedding): Embedding(4885, 256, padding_idx=0)
  (transformer_decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
     

INFO:pyhealth.trainer:TransformerEHRGenerator(
  (token_embedding): Embedding(4885, 256, padding_idx=0)
  (transformer_decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2)

Metrics: None


INFO:pyhealth.trainer:Metrics: None


Device: cuda


...

New best loss score (5.6858) at epoch-4, step-470


INFO:pyhealth.trainer:New best loss score (5.6858) at epoch-4, step-470


Loaded best model


INFO:pyhealth.trainer:Loaded best model



Training completed!


In [13]:
# Save the best model
torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
print(f"Model saved to: {config.MODEL_SAVE_PATH}")

Model saved to: /content/drive/MyDrive/PyHealth_Synthetic_EHR/models/transformer_ehr_best.pth


## 7. Evaluation on Test Set

In [14]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = trainer.evaluate(test_loader)

print("\nTest Results:")
for metric, value in test_results.items():
    print(f"  {metric}: {value:.4f}")

Evaluating on test set...


Evaluation:   0%|          | 0/12 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Evaluation: 100%|██████████| 12/12 [00:02<00:00,  5.78it/s]


Test Results:
  loss: 5.6757


## 8. Generate Synthetic Patients

In [15]:
# Generate synthetic patient histories
print(f"Generating {config.NUM_SYNTHETIC_SAMPLES} synthetic patients...\n")

model.eval()
with torch.no_grad():
    synthetic_nested_codes = model.generate(
        num_samples=config.NUM_SYNTHETIC_SAMPLES,
        max_visits=config.MAX_GEN_VISITS,
        max_codes_per_visit=config.MAX_CODES_PER_VISIT,
        temperature=config.TEMPERATURE,
        top_k=config.TOP_K,
        top_p=config.TOP_P
    )

print(f"Generated {len(synthetic_nested_codes)} synthetic patients")
print(f"\nExample synthetic patient (first 3 visits):")
if len(synthetic_nested_codes) > 0 and len(synthetic_nested_codes[0]) > 0:
    for i, visit in enumerate(synthetic_nested_codes[0][:3]):
        print(f"  Visit {i+1}: {visit[:10]}{'...' if len(visit) > 10 else ''}")

Generating 1000 synthetic patients...

Generated 1000 synthetic patients

Example synthetic patient (first 3 visits):
  Visit 1: [47]
  Visit 2: [273, 188, 1124, 1125, 1872]
  Visit 3: [1873]


## 9. Convert to DataFrame Format

In [16]:
import pandas as pd

# Get the processor to convert token IDs back to codes
input_processor = sample_dataset.input_processors["visit_codes"]
index_to_code = {v: k for k, v in input_processor.code_vocab.items()}

print("Converting synthetic data to CSV format...")

# Convert nested codes to tabular format
rows = []
for patient_idx, patient_visits in enumerate(synthetic_nested_codes):
    synthetic_subject_id = f"SYNTHETIC_{patient_idx:06d}"

    for visit_num, visit_codes in enumerate(patient_visits, start=1):
        for code_idx in visit_codes:
            # Convert token ID to actual code
            code = index_to_code.get(code_idx, str(code_idx))

            # Skip special tokens
            if code in ['<pad>', '<bos>', '<eos>', 'VISIT_DELIM']:
                continue

            rows.append({
                'SUBJECT_ID': synthetic_subject_id,
                'VISIT_NUM': visit_num,
                'ICD9_CODE': code
            })

# Create DataFrame
synthetic_df = pd.DataFrame(rows)

print(f"\nCreated DataFrame with {len(synthetic_df)} rows")
print(f"Number of unique patients: {synthetic_df['SUBJECT_ID'].nunique()}")
print(f"Number of unique codes: {synthetic_df['ICD9_CODE'].nunique()}")

# Display sample
print("\nSample of synthetic data:")
print(synthetic_df.head(20))

Converting synthetic data to CSV format...

Created DataFrame with 429124 rows
Number of unique patients: 1000
Number of unique codes: 365

Sample of synthetic data:
          SUBJECT_ID  VISIT_NUM ICD9_CODE
0   SYNTHETIC_000000          1      2967
1   SYNTHETIC_000000          2     07032
2   SYNTHETIC_000000          2      V053
3   SYNTHETIC_000000          2      7707
4   SYNTHETIC_000000          2     77181
5   SYNTHETIC_000000          2     76516
6   SYNTHETIC_000000          3     76526
7   SYNTHETIC_000000          4      7512
8   SYNTHETIC_000000          4      V053
9   SYNTHETIC_000000          5      V053
10  SYNTHETIC_000000          6     77181
11  SYNTHETIC_000000          6      7455
12  SYNTHETIC_000000          6      7742
13  SYNTHETIC_000000          6     77081
14  SYNTHETIC_000000          6      7776
15  SYNTHETIC_000000          7     76527
16  SYNTHETIC_000000          7     36221
17  SYNTHETIC_000000          8      V053
18  SYNTHETIC_000000          8     

## 10. Save CSV File

In [18]:
# Save to CSV
output_csv_path = f"{config.OUTPUT_DIR}/synthetic_ehr_transformer.csv"
synthetic_df.to_csv(output_csv_path, index=False)

print(f"Synthetic data saved to: {output_csv_path}")
print(f"\nFile info:")
print(f"  Rows: {len(synthetic_df):,}")
print(f"  Columns: {list(synthetic_df.columns)}")
print(f"  File size: {os.path.getsize(output_csv_path) / 1024:.2f} KB")

Synthetic data saved to: /content/drive/MyDrive/PyHealth_Synthetic_EHR/output/synthetic_ehr_transformer.csv

File info:
  Rows: 429,124
  Columns: ['SUBJECT_ID', 'VISIT_NUM', 'ICD9_CODE']
  File size: 10567.09 KB


## 11. Summary

In [19]:
# Print final summary
print("\n" + "="*60)
print("SYNTHETIC EHR GENERATION SUMMARY")
print("="*60)

print(f"\nModel: TransformerEHRGenerator")
print(f"  - Embedding dim: {config.EMBEDDING_DIM}")
print(f"  - Layers: {config.NUM_LAYERS}")
print(f"  - Attention heads: {config.NUM_HEADS}")
print(f"  - Parameters: {total_params:,}")

print(f"\nTraining:")
print(f"  - Epochs: {config.EPOCHS}")
print(f"  - Batch size: {config.BATCH_SIZE}")
print(f"  - Training samples: {len(train_dataset)}")
print(f"  - Validation samples: {len(val_dataset)}")

print(f"\nGeneration:")
print(f"  - Synthetic patients: {synthetic_df['SUBJECT_ID'].nunique()}")
print(f"  - Total diagnosis records: {len(synthetic_df)}")
print(f"  - Unique ICD-9 codes: {synthetic_df['ICD9_CODE'].nunique()}")
print(f"  - Avg visits per patient: {visits_per_patient.mean():.2f}")
print(f"  - Avg codes per visit: {codes_per_visit.mean():.2f}")

print(f"\nOutput:")
print(f"  - CSV file: {output_csv_path}")
print(f"  - Model checkpoint: {config.MODEL_SAVE_PATH}")

print("\n" + "="*60)
print("Pipeline completed successfully!")
print("="*60)


SYNTHETIC EHR GENERATION SUMMARY

Model: TransformerEHRGenerator
  - Embedding dim: 256
  - Layers: 6
  - Attention heads: 8
  - Parameters: 8,957,717

Training:
  - Epochs: 5
  - Batch size: 64
  - Training samples: 5999
  - Validation samples: 750

Generation:
  - Synthetic patients: 1000
  - Total diagnosis records: 429124
  - Unique ICD-9 codes: 365
  - Avg visits per patient: 66.81
  - Avg codes per visit: 6.42

Output:
  - CSV file: /content/drive/MyDrive/PyHealth_Synthetic_EHR/output/synthetic_ehr_transformer.csv
  - Model checkpoint: /content/drive/MyDrive/PyHealth_Synthetic_EHR/models/transformer_ehr_best.pth

Pipeline completed successfully!
